In this lab:

- Convolutional Neural Network from scratch
- Logging with WandB
- Checkpoints
- Early stopping


In [ ]:
import torch  # tensor operations and deep learning functionalities
import torchvision  # computer vision tasks
from torchvision import transforms as T  # image transformation utilities (e.g., resizing, normalization)
import torch.nn.functional as F  # activation and loss functions
from pathlib import Path  # to manage file paths

### Improved Training Loop

We will enhance the training loop discussed in the theory lectures by incorporating the following three steps:

**1) Logging:** We can save the hyperparameters associated with the current experiments. Printing out the results makes it difficult to organize different runs of the same experiment and compare the results.

However, some libraries can handle this for us. For instance, [TensorBoard](https://www.tensorflow.org/tensorboard). Unlike TensorBoard, we can also use [WandB](https://wandb.ai/site), which is very user-friendly since it is a hosted service—all we need to do is create an account, log in, and log our files.

**2) Save/Load Checkpoints:** When training models for a high number of epochs, it is always a good idea to save checkpoints of the model during the process. For example, the training may fail at a certain point, or we may observe strange training behavior. In this case, it is always convenient to have checkpoints, as they allow us to restart the training from a specific epoch or reproduce potential errors.

In PyTorch, we can easily save model with the comand `torch.save()` [doc](https://pytorch.org/docs/stable/generated/torch.save.html), this comands will save the weights of our model as dictionary in an output file.

We can load weights from the disk with the command `torch.load()`, which returns a dictionary with the assocaited weights.

❗Saving models to the disk may take a lot of space, be aware of that.

**3) Early Stopping:** Sometimes, we set a certain number of epochs $N$, but after a while we don't see major imporvements in the performance, in this case it may be a waste of resources to go on with the training and a better solution is to early stop our training loop.

In worst case scenarios, we can even see the preformance *decrease* as the training goes on. This is usually a warning of **overfitting**.

One of the strategies we can apply to reduce this risk is to **stop the training.**

🛠 There are libraries like **Pytorch-Lightninig**, that gives relevant methods out of the box (among with a lot of other useful things). But, maybe a little bit advanced, hence we are going to implement our own `early_stopping` function.

❗Don't kill the training too early

In [ ]:
!pip install wandb -qU

In [ ]:
import wandb
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: beyancigdem (cbeyan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Below, we will make a normal MLP training/evaluation without injecting the things listed above.


In [ ]:
def get_data(batch_size, test_batch_size=256):

  # Prepare data transformations and then combine them sequentially
  transform = list()
  transform.append( # [TO DO] )      # converts Numpy to Pytorch Tensor
  transform.append( # [TO DO] )      # Normalizes the Tensors between [-1, 1]: mean 0.5 and std=0.5
  transform = T.Compose(transform)   # Composes the above transformations into one.

  # Load data
  full_training_data = torchvision.datasets.MNIST('./data', train=True, transform=transform, download=True)
  test_data = torchvision.datasets.MNIST('./data', train=False, transform=transform, download=True)

  print(full_training_data)

  # Create train and validation splits
  num_samples = len(full_training_data)
  training_samples = int(# [TO DO] ) # 70% of the data
  validation_samples = # [TO DO]

  training_data, validation_data = torch.utils.data.random_split(full_training_data, [training_samples, validation_samples])

  # Initialize dataloaders
  train_loader = torch.utils.data.DataLoader(# [TO DO], num_workers=4)
  val_loader = torch.utils.data.DataLoader(# [TO DO], num_workers=4)
  test_loader = torch.utils.data.DataLoader(# [TO DO], num_workers=4)

  return train_loader, val_loader, test_loader


In [ ]:
train_loader, val_loader, test_loader = get_data(128)  # Call get_data function with batch size of 128, returns data loaders

i = iter(train_loader)  # Convert train_loader into an iterator to access batches
a = next(i)  # Retrieve the next batch from the train_loader iterator

for t in a:  # Iterate through each element in the batch (which contains images and labels)
    print(t.dtype)  # Print the data type (dtype) of each element in the batch (e.g., image or label)


100%|██████████| 9.91M/9.91M [00:00<00:00, 13.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 456kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.43MB/s]

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.5], std=[0.5])
           )



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


torch.float32
torch.int64


In [ ]:
# Our network
class MLP(torch.nn.Module):  # Define a class MLP that inherits from PyTorch's nn.Module
  def __init__(self, input_dim, hidden_dim, output_dim):  # Constructor to initialize dimensions for the layers
    super(MLP, self).__init__()  # Call the parent class constructor (nn.Module)

    # Define the layers of the network
    self.input_to_hidden = # [TO DO]  # Linear layer from input to hidden layer
    self.hidden_to_output = # [TO DO]  # Linear layer from hidden to output layer
    self.activation = # [TO DO]  # Sigmoid activation function to apply between layers

  def forward(self, x):  # Define the forward pass of the network
    # Puts the output in (batch_size, input_dim) format (flatten the input)
    x = x.view(x.shape[0], -1)  # Flatten the input tensor, preserving batch size

    # Forward input through the layers
    x = # [TO DO]  # Pass through the first layer (input to hidden)
    x = # [TO DO]  # Apply the activation function (Sigmoid)
    x = # [TO DO]  # Pass through the second layer (hidden to output)
    return x  # Return the output (logits or predictions)


In [ ]:
# Function to train the model for one epoch
def train_one_epoch(net, data_loader, optimizer, cost_function, device='cuda'):
  samples = 0.  # Variable to track the number of samples processed
  cumulative_loss = 0.  # Variable to accumulate the total loss for averaging later
  cumulative_accuracy = 0.  # Variable to accumulate the total accuracy for averaging later

  net.train()  # Set the network in training mode (important for layers like dropout or batch norm)

  # Loop over each batch in the data loader
  for batch_idx, (inputs, targets) in enumerate(data_loader):
    # Load data into GPU if available
    inputs = inputs.to(device)
    targets = targets.to(device)

    # Forward pass: pass inputs through the network
    outputs = net(inputs)

    # Calculate the loss using the cost function
    loss = cost_function(outputs, targets)

    # Backward pass: compute the gradients
    # [TO DO]

    # Update the model parameters using the optimizer
    # [TO DO]

    # Reset the gradients after the update
    # [TO DO]

    # Track the number of samples and the loss
    samples += inputs.shape[0]
    cumulative_loss += loss.item()  # .item() extracts the scalar value from the tensor

    # Get predicted class labels by taking the index of the maximum output value
    predicted = # [TO DO]

    # Track the number of correct predictions for accuracy calculation
    cumulative_accuracy += predicted.eq(targets).sum().item()

  # Return average loss and accuracy for this epoch
  return cumulative_loss / samples, cumulative_accuracy / samples * 100


# Validation function to evaluate the model
def validation_step(net, data_loader, cost_function, device='cuda'):
  samples = 0.  # Variable to track the number of samples processed
  cumulative_loss = 0.  # Variable to accumulate the total loss for averaging later
  cumulative_accuracy = 0.  # Variable to accumulate the total accuracy for averaging later

  net.eval()  # Set the network in evaluation mode (important for layers like dropout or batch norm)

  # Disable gradient calculation for validation to save memory and computation
  with torch.no_grad():
    # Loop over each batch in the data loader
    for batch_idx, (inputs, targets) in enumerate(data_loader):
      # Load data into GPU if available
      inputs = inputs.to(device)
      targets = targets.to(device)

      # Forward pass: pass inputs through the network
      outputs = # [TO DO]

      # Calculate the loss using the cost function
      loss = # [TO DO]

      # Track the number of samples and the loss
      samples += inputs.shape[0]
      cumulative_loss += loss.item()  # .item() extracts the scalar value from the tensor

      # Get predicted class labels by taking the index of the maximum output value
      _, predicted = outputs.max(1)

      # Track the number of correct predictions for accuracy calculation
      cumulative_accuracy += predicted.eq(targets).sum().item()

  # Return average loss and accuracy for the validation step
  return cumulative_loss / samples, cumulative_accuracy / samples * 100


# Function to get the optimizer (Stochastic Gradient Descent)
def get_optimizer(net, lr, wd, momentum):
  # Using SGD with learning rate, weight decay, and momentum
  optimizer = # [TO DO]
  return optimizer

# Function to get the cost function (Cross Entropy Loss for classification)
def get_cost_function():
  # Define the loss function used for multi-class classification
  cost_function = # [TO DO]
  return cost_function


Now, we modify the `main()` and add:

*   Logging
*   Early Stopping
*   Checkpoints saving

In [ ]:
def main(batch_size=128, input_dim=28*28, hidden_dim=100, output_dim=10, device='cuda:0', learning_rate=0.01, weight_decay=0.000001, momentum=0.9, epochs=10,
         checkpoint_path="./checkpoints", run_name="prova", network_architecture="mlp", save_every_n_epochs=2, patience=3):

  # ===============  ADDED  ====================
  # Logging
  # Initialize the wandb project and log hyperparameters and metrics
  wandb.init(
      # Set the project where this run will be logged
      project="Lab04",
      # Run name for easy tracking
      name=f"{run_name}",
      # Log hyperparameters and other metadata
      config={
      "learning_rate": learning_rate,
      "architecture": network_architecture,
      "dataset": "MNIST",
      "epochs": epochs,
      "weight_decay":weight_decay,
      "momentum":momentum,
      "batch_size":batch_size,
      })

  # Create the checkpoint directory
  checkpoint_path = Path(checkpoint_path)
  checkpoint_path = checkpoint_path / run_name
  checkpoint_path.mkdir(exist_ok=True, parents=True)  # Ensure the directory exists, create it if not

  # Early Stopping parameters
  patience = 3
  trigger = 0
  # ===========================================

  # Gets the training, validation, and test data loaders
  train_loader, val_loader, test_loader = get_data(batch_size)

  # Initialize the network and move it to the selected device (GPU)
  if network_architecture == # [TO DO]
    net = MLP(input_dim, hidden_dim, output_dim).to(device)
  elif network_architecture == # [TO DO]
    net = CNN().to(device)
  else:
    raise NotImplementedError("Choose either mlp or cnn")

  # Initialize the optimizer
  optimizer = # [TO DO]

  # Initialize the cost function (loss function)
  cost_function = # [TO DO]

  # ===============  ADDED  ====================
  # Keep track of the best validation accuracy and validation loss
  best_val_accuracy = 0.0
  best_val_loss = float('inf')
  # ============================================

  # Loop over each epoch to train the network and evaluate its performance
  for e in range(epochs):
    # Train for one epoch and compute training accuracy and loss
    train_loss, train_accuracy = train_one_epoch( # [TO DO]
    # Compute validation loss and accuracy
    val_loss, val_accuracy = validation_step( # [TO DO]

    # Print the results for the current epoch
    print('Epoch: {:d}'.format(e+1))
    print('\t Training loss {:.5f}, Training accuracy {:.2f}'.format(train_loss, train_accuracy))
    print('\t Validation loss {:.5f}, Validation accuracy {:.2f}'.format(val_loss, val_accuracy))
    print('\t Best Accuracy {:.5f}'.format(best_val_accuracy))
    print('-----------------------------------------------------')

    # ===============  ADDED  ====================
    # Log the training and validation stats to wandb
    wandb.log({
            "train/loss": train_loss,
            "train/accuracy": train_accuracy,
            "val/loss": val_loss,
            "val/accuracy": val_accuracy
        })

    # Save the model checkpoints after every n epochs or the last epoch
    if e % save_every_n_epochs == 0 or e == (epochs - 1):
      torch.save(net.state_dict(), checkpoint_path / f'epoch-{e}.pth')

    # Update the best model based on validation accuracy
    if val_accuracy >= best_val_accuracy:
      torch.save(net.state_dict(), checkpoint_path / f'best.pth')
      best_val_accuracy = # [TO DO]

    # Early stopping: stop training if validation loss does not improve
    if val_loss > best_val_loss:
      trigger += 1
      if trigger == patience:
        print(f"Validation Accuracy did not improve for {patience} epochs. Stopping training...")
        break
    else:
      best_val_loss = val_loss
      trigger = 0
    # ===========================================


Let's train an analyze the results

In [ ]:
# Call the main function with the run name "mlp_experiment_01" and set the number of epochs to 10 for training.
main(run_name="mlp_experiment_01", epochs=10)

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.5], std=[0.5])
           )
Epoch: 1
	 Training loss 0.00853, Training accuracy 73.61
	 Validation loss 0.00210, Validation accuracy 87.23
	 Best Accuracy 0.00000
-----------------------------------------------------
Epoch: 2
	 Training loss 0.00340, Training accuracy 88.53
	 Validation loss 0.00153, Validation accuracy 89.38
	 Best Accuracy 87.22707
-----------------------------------------------------
Epoch: 3
	 Training loss 0.00275, Training accuracy 90.21
	 Validation loss 0.00132, Validation accuracy 90.42
	 Best Accuracy 89.37719
-----------------------------------------------------
Epoch: 4
	 Training loss 0.00246, Training accuracy 91.05
	 Validation loss 0.00122, Validation accuracy 91.03
	 Best Accuracy 90.42169
-----------------------------------------------------
Epoch: 5
	 Training los

In [ ]:
model = MLP(input_dim=28*28, hidden_dim=100, output_dim=10) #create our model as MLP

In [ ]:
# Load the saved model checkpoint from the file 'best.pth' located in the directory './checkpoints/mlp_experiment_01/'
# The 'map_location="cpu"' argument ensures that the model is loaded onto the CPU, even if it was originally saved on a GPU
ckpt = torch.load("./checkpoints/mlp_experiment_01/best.pth", map_location="cpu")


In [ ]:
# Load the model's state dictionary (weights and biases) from the checkpoint into the model
# This restores the trained parameters of the model saved in the checkpoint
model.load_state_dict(ckpt)


<All keys matched successfully>

----------------------------------------------

## CNN - The LeNet

We will define a **LeNet** architecture, shown as follows.

<img src="https://d2l.ai/_images/lenet.svg" width="900"></br></br>

The main components are:

**Convolutional Layer**: The convolutional layers can be simply defined using `torch.nn.Conv2d` module of `torch.nn` package.

<img src="https://miro.medium.com/v2/resize:fit:790/1*1VJDP6qDY9-ExTuQVEOlVg.gif" width="200" calss="center"></br></br>


**Pooling Layer:** Pooling operation to reduce the size of convolutional feature maps. For this case we are going to use `torch.nn.functional.max_pool2d`. Another common choice is to use average pooling.
<img src="https://nico-curti.github.io/NumPyNet/NumPyNet/images/maxpool.gif" width="250" calss="center"></br></br>

**Fully Conected/Activation:**
Differently from the MLP example, we will use a Rectified Linear Units (ReLU) as activation function with the help of `torch.nn.functional.relu`, replacing `torch.nn.Sigmoid`.

In [ ]:
class CNN(torch.nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    # Conv2d
    # input channel = 1, output channels = 6, kernel size = 5
    # input image size = (28, 28), image output size = (24, 24)
    self.conv1 = # [TO DO]
    self.act1 = torch.nn.ReLU()
    self.max_pool1 = # [TO DO] # max pooling kernel size = 2

    # Conv2d
    # input channel = 6, output channels = 16, kernel size = 5
    # input image size = (12, 12), output image size = (8, 8)
    self.conv2 = # [TO DO]
    self.act2 = torch.nn.ReLU()
    self.max_pool2 = # [TO DO] # max pooling kernel size = 2

    # Linear Layer
    # input dim = 4 * 4 * 16 ( H x W x C), output dim = 120
    self.fc3 = # [TO DO]
    self.act3 = torch.nn.ReLU()

    # Linear Layer
    # input dim = 120, output dim = 84
    self.fc4 = # [TO DO]
    self.act4 = torch.nn.ReLU()

    # Linear Layer
    # input dim = 84, output dim = 10
    self.fc5 = # [TO DO]

  def forward(self, x):

    x = # [TO DO]
    x = # [TO DO]
    # Max Pooling with kernel size = 2
    # output size = (12, 12)
    x = # [TO DO]

    x = # [TO DO]
    x = # [TO DO]
    # Max Pooling with kernel size = 2
    # output size = (4, 4)
    x = # [TO DO]

    # From now on, it is similar to the MLP
    # flatten the feature maps into a long vector
    x = x.view(x.shape[0], -1)
    #(bs, 4*4*16)
    x = # [TO DO]
    x = # [TO DO]

    x = # [TO DO]
    x = # [TO DO]

    x = # [TO DO]

    return x

Let's train!

In [ ]:
main(run_name="cnn_experiment_02", network_architecture="cnn", epochs=20)

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=[0.5], std=[0.5])
           )
Epoch: 1
	 Training loss 0.00760, Training accuracy 69.39
	 Validation loss 0.00068, Validation accuracy 94.73
	 Best Accuracy 0.00000
-----------------------------------------------------
Epoch: 2
	 Training loss 0.00093, Training accuracy 96.42
	 Validation loss 0.00041, Validation accuracy 96.62
	 Best Accuracy 94.72748
-----------------------------------------------------
Epoch: 3
	 Training loss 0.00065, Training accuracy 97.47
	 Validation loss 0.00034, Validation accuracy 97.18
	 Best Accuracy 96.62203
-----------------------------------------------------
Epoch: 4
	 Training loss 0.00053, Training accuracy 97.93
	 Validation loss 0.00025, Validation accuracy 97.95
	 Best Accuracy 97.18318
-----------------------------------------------------
Epoch: 5
	 Training los